# MEC_53614 — Artificial intelligence for solids and structures — Martin Genet & Ioannis Stefanou

# L1 — Introduction: one tensile-test campaign seen through the four poles

# E1 — Learning from a tensile-test campaign

In [ ]:
# Names: …

Let us start with some imports…

In [ ]:
# Computing libraries
import numpy   as np     # doc: https://numpy.org/doc/stable/reference
import sklearn as skl    # doc: https://scikit-learn.org/stable/api/index.html

# Plotting libraries
import matplotlib.pyplot as mpl    # doc: https://matplotlib.org/stable/contents.html

## Colab only: install local library
# !pip install git+https://github.com/mgenet/LIB53614.git

# Course library
import LIB53614 as lib

## Problem

A laboratory has run a tensile-test campaign on several materials: three specimens of each of six materials: a structural steel, an aluminium alloy, a copper, a cast aluminium alloy, a grey cast iron, and a polymer (PMMA).
For each specimen, we have the measured curve (engineering strain $\varepsilon$, engineering stress $\sigma$), and the fracture mode observed on the fracture surface (ductile or brittle).

We will ask four questions on this campaign, one for each of the four "poles" of the course:
* **Regression:** what is the Young modulus of each material?
* **Classification:** is a material ductile or brittle?
* **Free-form models:** can a model learned from data replace a mechanical model? (On a cantilever beam, in part D.)
* **LLMs / agents:** can we let an AI assistant write our post-processing scripts? (Part E.)

The data are synthetic: they are generated by the course library, `LIB53614`, from mechanical models and a realistic measurement process.
It includes random errors (noise on the force and on the strain), but also systematic errors (biases) due to the test procedure: for instance, as in real tests, the specimen is slightly loaded, to take up the slack in the grips, before the strain is zeroed.
The parameters actually used to generate each curve (the "truth") can be retrieved with `lib.tensile.truth(specimen)`.
This allows us to check our methods, which is rarely possible with real data—use it only to evaluate a method, never inside the method itself!

## A. Exploring the campaign

We first generate the campaign, and print the list of specimens.

In [ ]:
specimens = lib.tensile.campaign(n_per_material=3, seed=0)
for specimen in specimens:
    print(f"{specimen.name:12s} {lib.tensile.MATERIALS[specimen.material]['label']:30s} {len(specimen.strain):5d} points, fracture: {specimen.fracture}")

We now plot all the curves, with two zooms on their beginning: on the elastic part (middle), and on the very first points (right).

In [ ]:
colors = dict(zip(lib.tensile.MATERIALS.keys(), mpl.rcParams["axes.prop_cycle"].by_key()["color"]))
fig, axes = mpl.subplots(1, 3, figsize=(21, 5))
for specimen in specimens:
    for ax in axes:
        ax.plot(100*specimen.strain, specimen.stress, color=colors[specimen.material], lw=1, marker=("." if (ax is axes[2]) else None), ms=4, label=specimen.material if specimen.name.endswith("01") else None)
for ax in axes:
    ax.set_xlabel("engineering strain (%)")
    ax.set_ylabel("engineering stress (MPa)")
    ax.grid()
axes[0].legend()
axes[0].set_xlim(-2.   , +52.  );
axes[0].set_ylim(-20.  , +600. );
axes[1].set_xlim(-0.02 , +0.52 );
axes[1].set_ylim(-20.  , +500. );
axes[2].set_xlim(-0.002, +0.022);
axes[2].set_ylim(-2.   , +52.  );

**Q1. In the vocabulary of machine learning, when we identify the Young modulus of a material from these curves, what are the data, the input (feature), the output (target), the model, and its parameter(s)?**

**Q2. Look at the zoom on the very first points (right plot). Do the curves start at the origin? Why?**

## B. Pole 1 — Regression: identifying the Young modulus

### B.1. Least squares

We start by identifying the Young modulus by using the elastic part of the curve.
Let us start with the Hooke law, i.e., find $E$ such that $\sigma = E \varepsilon$.

**Q3. Write the least-squares cost function for identifying $E$ from $N$ points $\left(\varepsilon_i, \sigma_i\right)$, and show that its minimizer is $E^* = \sum_i \varepsilon_i \sigma_i / \sum_i \varepsilon_i^2$.**

**Q4. Complete and execute the following code, which identifies $E$ from the points of a stress window $\left[\sigma_{min}, \sigma_{max}\right]$ (i) with your formula, (ii) with `np.linalg.lstsq`, and (iii) with `scikit-learn`'s `LinearRegression`.
What do you get for the first specimen? How do the three estimates compare with each other, and with the true value? Why?**

Hints:
* `mask = (V >= a) & (V <= b)` returns the boolean array that is `True` where `a ≤ V ≤ b`, and `V[mask]` returns the corresponding components.
* `np.sum(V)` returns the sum of the components of `V`.
* The model can be written `sig = M.X`, where `X = [E]` contains the parameter, and `M` is a 2D array with one row per point and one column per parameter: here, a single column, `eps`.
* `V[:,None]` turns the 1D array `V` into a 2D array with a single column.
* `np.linalg.lstsq(M, V)[0]` returns the least-squares solution `X` of `M.X = V`, an array with one component per column of `M`.
* `skl.linear_model.LinearRegression(fit_intercept=False).fit(X, y)` fits the model `y = X.coef_` (`X` is a 2D array, one column per feature), and returns the fitted model; its parameters are then in `model.coef_` (an array, one component per feature).
* Remember Q2.

In [ ]:
# Here we define the function
def identify_E_origin(strain, stress, sigma_min, sigma_max, method="sklearn"):
    mask = ### YOUR CODE HERE ###
    eps = strain[mask]; sig = stress[mask]
    if (method == "formula"):
        E = ### YOUR CODE HERE ###
    elif (method == "numpy"):
        M = ### YOUR CODE HERE ###
        E = ### YOUR CODE HERE ###
    elif (method == "sklearn"):
        model = ### YOUR CODE HERE ###
        E = ### YOUR CODE HERE ###
    return E

# Here we test the function
specimen = specimens[0]
for method in ["formula", "numpy", "sklearn"]:
    E = identify_E_origin(specimen.strain, specimen.stress, sigma_min=0., sigma_max=150., method=method)
    print(f"{method:7s}: E = {E:.0f} MPa")
print(f"true   : E = {lib.tensile.truth(specimen)['E']:.0f} MPa")

**Q5. To take into account this measurement bias, we now fit a straight line, $\sigma = E \varepsilon + \sigma_0$.
Write the least-squares cost function for identifying $\left(E, \sigma_0\right)$, and show that its minimizer is $E^* = \sum_i \left(\varepsilon_i - \bar{\varepsilon}\right) \left(\sigma_i - \bar{\sigma}\right) / \sum_i \left(\varepsilon_i - \bar{\varepsilon}\right)^2$ and $\sigma_0^* = \bar{\sigma} - E^* \bar{\varepsilon}$, where $\bar{\varepsilon}$ and $\bar{\sigma}$ are the means of the $\varepsilon_i$ and of the $\sigma_i$.**

**Q6. Complete and execute the following code, which identifies $E$ and $\sigma_0$ with the same three methods.
How do the estimates compare with the true values?**

Hints:
* `np.mean(V)` returns the mean of the components of `V`.
* The model can now be written `sig = M.X`, with `X = [E, sigma_0]`: `M` has a second column, filled with ones.
* `np.stack([V, W], axis=1)` returns the 2D array whose columns are `V` and `W`, and `np.ones_like(V)` returns an array of ones with the shape of `V`.
* `skl.linear_model.LinearRegression(fit_intercept=True)` fits the model `y = X.coef_ + intercept_`; the intercept is then in `model.intercept_` (a number).

In [ ]:
# Here we define the function
def identify_E(strain, stress, sigma_min, sigma_max, method="sklearn"):
    mask = (stress >= sigma_min) & (stress <= sigma_max)
    eps = strain[mask]; sig = stress[mask]
    if (method == "formula"):
        E       = ### YOUR CODE HERE ###
        sigma_0 = ### YOUR CODE HERE ###
    elif (method == "numpy"):
        M = ### YOUR CODE HERE ###
        E, sigma_0 = ### YOUR CODE HERE ###
    elif (method == "sklearn"):
        model = ### YOUR CODE HERE ###
        E, sigma_0 = ### YOUR CODE HERE ###
    return E, sigma_0

# Here we test the function
for method in ["formula", "numpy", "sklearn"]:
    E, sigma_0 = identify_E(specimen.strain, specimen.stress, sigma_min=0., sigma_max=150., method=method)
    print(f"{method:7s}: E = {E:.0f} MPa, sigma_0 = {sigma_0:.2f} MPa")
print(f"true   : E = {lib.tensile.truth(specimen)['E']:.0f} MPa, sigma_0 = {lib.tensile.truth(specimen)['preload']:.2f} MPa")

### B.2. Which points? (Model selection)

The result depends on the stress window used for the fit.

**Q7. Complete and execute the following code, which plots the relative error on $E$ as a function of the upper bound of the window, $\sigma_{max}$ (with $\sigma_{min} = 0$), for three specimens.
For each specimen, what is the best $\sigma_{max}$ (relative to the tensile strength $R_m$), and the corresponding error?
What limits the window from below? From above?**

In [ ]:
for name in ["S355-01", "Cu-01", "PMMA-01"]:
    specimen = [s for s in specimens if (s.name == name)][0]
    E_true = lib.tensile.truth(specimen)["E"]
    R_m = specimen.stress.max()
    sigma_max_lst = np.linspace(0.06, 1., 95) * R_m
    error_lst = []
    for sigma_max in sigma_max_lst:
        E, _ = identify_E(specimen.strain, specimen.stress, sigma_min=0., sigma_max=sigma_max)
        ### YOUR CODE HERE ###
    mpl.plot(sigma_max_lst/R_m, 100*np.array(error_lst), label=name)
mpl.axhline(0., color="k", lw=1)
mpl.xlabel("σ_max / R_m")
mpl.ylabel("relative error on E (%)")
mpl.ylim(-5., +5.)
mpl.grid()
mpl.legend();

**Q8. Propose a rule to choose the window automatically, i.e., without knowing the true modulus, and implement it in the following function.
Test it on all the specimens of the campaign: what is the largest relative error, and for which material?**

Hints:
* Testing standards prescribe a linear regression on a stress range well inside the elastic domain; for metals, typically between about 10% and 40% of the yield strength.
But the yield strength is not known before the modulus…
* The 0.2% offset yield strength $R_{p0.2}$ is the stress at which the curve crosses the line $\sigma = \sigma_0 + E \left(\varepsilon - 0.002\right)$.
* `np.flatnonzero(mask)` returns the indices where the boolean array `mask` is `True`.

In [ ]:
# Here we define the function
def identify_E_auto(strain, stress):
    ### YOUR CODE HERE ###
    return E

# Here we test the function
error_lst = []
for specimen in specimens:
    E = identify_E_auto(specimen.strain, specimen.stress)
    error = E/lib.tensile.truth(specimen)["E"] - 1
    error_lst.append(error)
    print(f"{specimen.name:12s} E = {E:9.0f} MPa, relative error = {100*error:+6.2f}%")
print(f"largest relative error: {100*np.max(np.abs(error_lst)):.2f}%")

### B.3. From specimens to materials

**Q9. Complete and execute the following code, which computes, for each material, the mean and standard deviation of the identified moduli, and of the true moduli.
Is the spread you observe within a material due to your identification method, or to the material itself?
What would change with one specimen per material? With thirty?**

Hints:
* `[f(x) for x in X if g(x)]` returns the list of the `f(x)` for the elements `x` of `X` such that `g(x)` is `True`.
* `np.mean(V)` and `np.std(V)` return the mean and the standard deviation of the components of `V`.

In [ ]:
for material in lib.tensile.MATERIALS:
    E_lst      = ### YOUR CODE HERE ###
    E_true_lst = ### YOUR CODE HERE ###
    print(f"{material:9s} identified: {np.mean(E_lst):8.0f} ± {np.std(E_lst):5.0f} MPa, true: {np.mean(E_true_lst):8.0f} ± {np.std(E_true_lst):5.0f} MPa, nominal: {lib.tensile.MATERIALS[material]['E']:8.0f} MPa")

**Q10. For which materials is the identification hardest? Why? What does "the Young modulus" mean for a material whose curve is never linear?**

Hint:
* For plastics, the standard (ISO 527) defines the modulus as the slope of the chord between $\varepsilon = 0.05\%$ and $\varepsilon = 0.25\%$.

## C. Pole 2 — Classification: ductile or brittle?

The fracture mode of each specimen has been determined by examining its fracture surface; here, "ductile" means that necking occurred before fracture.
In practice, a common rule of thumb is that a material is ductile if its elongation at fracture $A$ is larger than 5%.
We now consider a larger campaign, with twenty specimens of each material.

**Q11. Complete and execute the following code, which computes the elongation at fracture of each specimen (which we approximate here by the last measured strain), and plots it for each material.**

In [ ]:
specimens_c = lib.tensile.campaign(n_per_material=20, seed=1)
A = ### YOUR CODE HERE ###
y = np.array([(s.fracture == "ductile") for s in specimens_c], dtype=int) # 1: ductile, 0: brittle
materials = list(lib.tensile.MATERIALS.keys())
k_material = np.array([materials.index(s.material) for s in specimens_c])
mpl.scatter(A, k_material + 0.15*np.random.default_rng(0).standard_normal(len(A)), c=y, cmap="coolwarm", s=15)
mpl.axvline(0.05, color="k", ls="--", label="A = 5%")
mpl.yticks(range(len(materials)), materials)
mpl.xscale("log")
mpl.xlabel("elongation at fracture A (-)")
mpl.title("red: ductile, blue: brittle")
mpl.legend();

**Q12. Complete and execute the following code, which applies the rule of thumb.
What fraction of the specimens is correctly classified? Which ones are misclassified?**

In [ ]:
y_rule = ### YOUR CODE HERE ###
print(f"accuracy: {np.mean(y_rule == y):.3f}")
print("misclassified:", [s.name for s, ok in zip(specimens_c, y_rule == y) if not ok])

**Q13. A threshold rule is a decision tree of depth one.
Complete and execute the following code, which learns such a tree from the data.
What threshold does it learn, and with which accuracy?
Why is it different from 5%?
What happens if the A356 specimens are removed from the training data?**

Hints:
* `skl.tree.DecisionTreeClassifier(max_depth=1).fit(X, y)` fits a tree of depth one (`X` is a 2D array, one column per feature), and returns it; its threshold is then in `tree.tree_.threshold[0]`.
* `tree.score(X, y)` returns the accuracy of the tree on the data `(X, y)`.

In [ ]:
tree = ### YOUR CODE HERE ###
print(f"all specimens:  threshold = {tree.tree_.threshold[0]:.4f}, accuracy = {tree.score(A[:,None], y):.3f}")

mask = np.array([(s.material != "A356") for s in specimens_c])
tree_wo = ### YOUR CODE HERE ###
print(f"without A356:   threshold = {tree_wo.tree_.threshold[0]:.4f}, accuracy on all specimens = {tree_wo.score(A[:,None], y):.3f}")

**Q14. Classification is regression with piecewise-constant models.
Complete and execute the following code, which fits a regression tree of depth one on the labels (0 or 1), and a logistic regression on $\log A$, and plots both as functions of $A$.
How do they compare with the classification tree of Q13?
What do they predict for $A = 7\%$?**

Hints:
* `skl.tree.DecisionTreeRegressor(max_depth=1)` and `skl.linear_model.LogisticRegression()` work like the classification tree.
* `model.predict(X)` returns the predictions of the model; for the logistic regression, `model.predict_proba(X)[:,1]` returns the probability of class 1.

In [ ]:
reg_tree = ### YOUR CODE HERE ###
log_reg  = ### YOUR CODE HERE ###

A_plot = np.geomspace(A.min(), A.max(), 1000)
mpl.plot(A, y, "k.", alpha=0.3, label="data")
mpl.plot(A_plot, reg_tree.predict(A_plot[:,None]), label="regression tree (depth 1)")
mpl.plot(A_plot, log_reg.predict_proba(np.log(A_plot)[:,None])[:,1], label="logistic regression")
mpl.xscale("log")
mpl.xlabel("elongation at fracture A (-)")
mpl.ylabel("ductile (1) / brittle (0)")
mpl.grid()
mpl.legend();

print(f"A = 7%: regression tree: {reg_tree.predict([[0.07]])[0]:.3f}, logistic regression: {log_reg.predict_proba(np.log([[0.07]]))[0,1]:.3f}")

## D. Pole 3 — Free-form models… or physics? The cantilever

We now consider a cantilever beam of length $L$, with a rectangular cross-section $b \times h$, made of a material of Young modulus $E$, and loaded by a force $P$ at its tip.
We want a model that predicts the tip deflection $\delta$ from $\left(P, L, E, b, h\right)$, and we have a database of deflections (computed, or measured, with a 2% noise) for beams in the following ranges: $P \in [1, 10]$ N, $L \in [0.1, 0.5]$ m, $E \in [70, 210]$ GPa, $b \in [1, 5]$ cm, $h \in [1, 3]$ cm.

<div style="text-align:center">
    
</div>

We generate a test set in the same ranges as the training data (to assess interpolation), and an "extrapolation" set with longer beams, $L \in [0.5, 1]$ m.

In [ ]:
X_test , delta_test  = lib.cantilever.samples(n=1000, seed=1)
X_extra, delta_extra = lib.cantilever.samples(n=1000, ranges={"L":(0.5, 1.0)}, seed=2)
print(f"test set:          delta in [{delta_test.min():.1e}, {delta_test.max():.1e}] m")
print(f"extrapolation set: delta in [{delta_extra.min():.1e}, {delta_extra.max():.1e}] m")

**Q15. Complete the following code, which computes the median relative error of a set of predictions.**

In [ ]:
def median_relative_error(delta_pred, delta):
    return ### YOUR CODE HERE ###

### D.1. A free-form model

**Q16. Execute the following code, which trains a neural network (a multilayer perceptron, more on this in L5) on the raw inputs, for an increasing number of training samples, and evaluates it on the test and extrapolation sets.
What are the coefficients of determination $R^2$ and the median relative errors for 1000 training samples?
Why are these two measures so different?**

Hint:
* $R^2 = 1 - \sum_i \left(\delta^{pred}_i - \delta_i\right)^2 / \sum_i \left(\delta_i - \bar{\delta}\right)^2$ is the score returned by `model.score(X, delta)`.

In [ ]:
def train_mlp(X, delta):
    model = skl.compose.TransformedTargetRegressor(
        regressor=skl.pipeline.make_pipeline(
            skl.preprocessing.StandardScaler(),
            skl.neural_network.MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=5000, random_state=0)),
        transformer=skl.preprocessing.StandardScaler())
    model.fit(X, delta)
    return model

for n in [20, 100, 1000, 10000]:
    X_train, delta_train = lib.cantilever.samples(n=n, seed=0)
    model = train_mlp(X_train, delta_train)
    print(f"n = {n:5d}:"
          f" test: R² = {model.score(X_test, delta_test):6.3f}, median relative error = {100*median_relative_error(model.predict(X_test), delta_test):5.1f}%;"
          f" extrapolation: R² = {model.score(X_extra, delta_extra):6.3f}, median relative error = {100*median_relative_error(model.predict(X_extra), delta_extra):5.1f}%")

### D.2. Physics in the features

Let us now assume that we only know that the deflection is a power law of the inputs (a common first guess in physics, which we will refine in D.3):
$$
    \delta = c \, P^{a_1} L^{a_2} E^{a_3} b^{a_4} h^{a_5}.
$$

**Q17. Show that the identification of $\left(c, a_1, \dots, a_5\right)$ is a linear regression problem.**

**Q18. Complete and execute the following code, which performs this regression from only 20 training samples.
Which exponents do you find? What are the median relative errors on the test and extrapolation sets?
Compare with the neural network of Q16.**

In [ ]:
X_train, delta_train = lib.cantilever.samples(n=20, seed=0)
model = ### YOUR CODE HERE ###
print("exponents:", np.round(model.coef_, 3), f"constant: {np.exp(model.intercept_):.3f}")
delta_pred_test  = ### YOUR CODE HERE ###
delta_pred_extra = ### YOUR CODE HERE ###
print(f"median relative error: {100*median_relative_error(delta_pred_test , delta_test ):.2f}% (test), {100*median_relative_error(delta_pred_extra, delta_extra):.2f}% (extrapolation)")

**Q19. The constant $c$ is poorly identified, yet the predictions are accurate. Why?
Complete and execute the following code, which rounds the exponents to the nearest integers (is it legitimate?), and identifies the constant only.
Compare with beam theory.**

In [ ]:
exponents = ### YOUR CODE HERE ###
c = ### YOUR CODE HERE ###
print("exponents:", exponents, f"constant: {c:.3f}")

### D.3. Dimensional analysis

The problem involves six variables, $\left(\delta, P, L, E, b, h\right)$, and two independent dimensions (force and length).
According to the Vaschy–Buckingham theorem, it can therefore be written in terms of four dimensionless groups, for instance:
$$
    \Pi_0 = \frac{\delta}{L}, \quad \Pi_1 = \frac{P}{E L^2}, \quad \Pi_2 = \frac{b}{L}, \quad \Pi_3 = \frac{h}{L},
    \quad \text{with} \quad \Pi_0 = f\left(\Pi_1, \Pi_2, \Pi_3\right).
$$

**Q20. Check that these groups are dimensionless. Assuming that $f$ is a power law, how many parameters remain to be identified?**

**Q21. Complete and execute the following code, which identifies them from only 5 training samples. What do you find?**

In [ ]:
def pi_groups(X):
    P, L, E, b, h = X.T
    return ### YOUR CODE HERE ###

X_train, delta_train = lib.cantilever.samples(n=5, seed=0)
model = ### YOUR CODE HERE ###
print("exponents:", np.round(model.coef_, 3), f"constant: {np.exp(model.intercept_):.3f}")

**Q22. So, when would a model learned from data be justified for the deflection of a beam? Give two concrete situations.**

## E. Pole 4 — LLMs: use them, then verify

Engineers increasingly ask large language models (LLMs) to write their scripts.
This is fine, as long as the results are verified—which is our job.

**Q23. Ask an LLM to write a Python function `postprocess(strain, stress)` that returns the Young modulus $E$, the 0.2% offset yield strength $R_{p0.2}$, the tensile strength $R_m$ and the elongation at fracture $A$ from the raw data of a tensile test.
Paste it in the following cell, without modifying it, and indicate the LLM and the prompt(s) you used.**

In [ ]:
# LLM: …
# Prompt(s): …
### YOUR CODE HERE ###

**Q24. Write a test that compares the results of this function with the truth, on all the specimens of the campaign (including the brittle ones, the borderline alloy, and a campaign without extensometer, cf. the bonus).
List every problem you find, explain how you detected it, and fix the function.**

In [ ]:
### YOUR CODE HERE ###

**Q25. Ask at least two different LLMs (a) the tip deflection of a cantilever under a tip load, and (b) the critical buckling load of a column clamped at its base and free at its top.
Do they agree? Check their answers with dimensional analysis, limiting cases, and, for (a), the data of part D.**

## Bonus

In many laboratories, the strain is not measured with an extensometer, but computed from the displacement of the crosshead of the testing machine.
This displacement includes the deformation of the machine itself (frame, grips, etc.), which is in series with the specimen.

**Q26. Execute the following code, which generates a campaign without extensometer, with four gauge lengths, $L_0 \in \{25, 50, 100, 200\}$ mm, and identifies the moduli with your automatic rule.
How large are the errors?
Would you have detected the problem without knowing the truth, e.g., from the quality of the fit?**

In [ ]:
specimens_ce = lib.tensile.campaign(n_per_material=3, materials=["S355", "Al6061"], seed=1, extensometer=False, gauge_lengths=(25., 50., 100., 200.))
for specimen in specimens_ce:
    E_app = identify_E_auto(specimen.strain, specimen.stress)
    print(f"{specimen.name:12s} L_0 = {specimen.gauge_length:5.1f} mm, E_app = {E_app:9.0f} MPa, relative error = {100*(E_app/lib.tensile.truth(specimen)['E']-1):+6.2f}%")

**Q27. Show that, if the machine has a compliance $C_m$ (displacement per unit force), the apparent modulus satisfies $1/E_{app} = 1/E + S C_m / L_0$, where $S$ is the cross-section area of the specimen.
Complete and execute the following code, which uses this relation to identify $E$ and $C_m$ for each material.
What do you find?
With only two gauge lengths, how would you choose them?**

In [ ]:
for material in ["S355", "Al6061"]:
    sp = [s for s in specimens_ce if (s.material == material)]
    inv_L_0   = ### YOUR CODE HERE ###
    inv_E_app = ### YOUR CODE HERE ###
    model = ### YOUR CODE HERE ###
    E   = ### YOUR CODE HERE ###
    C_m = ### YOUR CODE HERE ###
    E_true = np.mean([lib.tensile.truth(s)["E"] for s in sp])
    print(f"{material:8s} E = {E:9.0f} MPa (mean true value: {E_true:9.0f} MPa), C_m = {C_m:.2e} mm/N")

## AI usage statement

Please indicate in the following cell which AI tools you used (besides part E), for what, and how you checked their outputs.

In [ ]:
# AI usage: …